# LeetCode #163 — Missing Ranges: tìm các khoảng còn thiếu bằng "lính canh"

**Mức độ:** Medium (bài Premium) · **Chủ đề:** Array

Skeleton trên LeetCode:

```python
class Solution:
    def findMissingRanges(self, nums: List[int], lower: int, upper: int) -> List[List[int]]:
```

Mục tiêu của notebook:

1. Đọc đúng đề: các số **không** có trong `nums` nhưng thuộc `[lower, upper]` được gom thành **khoảng liên tiếp**.
2. Nhận ra khoảng trống nằm **giữa hai số liền kề** của `nums` — và ở **hai đầu**.
3. Kỹ thuật **lính canh** (sentinel): thêm `lower − 1` vào đầu và `upper + 1` vào cuối để hai đầu được xử lý như mọi chỗ khác.
4. Tránh lặp qua cả khoảng `[lower, upper]` (có thể dài `2·10^9`).

# 1. Bài toán là gì?

Cho khoảng đóng `[lower, upper]` và mảng `nums` **đã sắp xếp**, các phần tử **khác nhau**, đều nằm trong `[lower, upper]`.

Một số `x` gọi là **bị thiếu** nếu `lower <= x <= upper` và `x` không có trong `nums`.

Trả về danh sách **ngắn nhất** các khoảng `[a, b]` (đã sắp xếp) phủ **đúng** mọi số bị thiếu: mỗi số bị thiếu nằm trong đúng một khoảng, và không khoảng nào chứa số của `nums`.

**Constraints (theo đề):**

- `-10^9 <= lower <= upper <= 10^9`
- `0 <= nums.length <= 100`
- `lower <= nums[i] <= upper`
- Mọi phần tử của `nums` khác nhau.

(Phiên bản cũ của đề trả về chuỗi như `"2"`, `"4->49"` — xem bài tập 3.)

---

## Ví dụ

```text
nums = [0, 1, 3, 50, 75], lower = 0, upper = 99
->  [[2, 2], [4, 49], [51, 74], [76, 99]]

nums = [-1], lower = -1, upper = -1
->  []            (không thiếu số nào)

nums = [], lower = 1, upper = 1
->  [[1, 1]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bài toán khoảng (intervals)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bài toán khoảng (intervals)

### Biểu diễn và điều kiện chồng nhau

Một khoảng `[s, e]` gồm điểm đầu `s` và điểm cuối `e` (`s <= e`). Hai khoảng `[a, b]` và `[c, d]` **chồng nhau** (giao khác rỗng, tính cả chạm đầu mút) khi và chỉ khi:

```text
a <= d  and  c <= b          (cách nhớ: "mỗi khoảng bắt đầu trước khi khoảng kia kết thúc")

không chồng:  [a────b]            [c────d]       b < c
chồng:        [a────────b]
                    [c────────d]                 a <= d và c <= b
```

Phần giao (nếu có) là `[max(a, c), min(b, d)]`; hợp khi chồng là `[min(a, c), max(b, d)]`.

In [1]:
def chong_nhau(x, y):
    return x[0] <= y[1] and y[0] <= x[1]

for x_iv, y_iv in [([1, 3], [2, 6]), ([1, 2], [3, 4]), ([1, 4], [4, 5]), ([2, 3], [1, 10])]:
    giao = [max(x_iv[0], y_iv[0]), min(x_iv[1], y_iv[1])] if chong_nhau(x_iv, y_iv) else None
    print(f"{x_iv} và {y_iv}: chồng? {chong_nhau(x_iv, y_iv)} | giao: {giao}")

[1, 3] và [2, 6]: chồng? True | giao: [2, 3]
[1, 2] và [3, 4]: chồng? False | giao: None
[1, 4] và [4, 5]: chồng? True | giao: [4, 4]
[2, 3] và [1, 10]: chồng? True | giao: [2, 3]


### Sắp xếp theo điểm đầu

Hầu hết bài khoảng bắt đầu bằng **sắp xếp theo điểm đầu**. Sau đó, các khoảng có thể gộp với nhau luôn **liền kề** trong thứ tự: chỉ cần so khoảng hiện tại với khoảng **cuối cùng** đã gộp.

```text
sắp xếp: [1,3] [2,6] [8,10] [15,18]
[1,3] + [2,6] chồng → [1,6];  [8,10] không chồng [1,6] → khoảng mới; …
```

In [2]:
def gop(intervals):
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)       # gộp vào khoảng cuối
        else:
            out.append([s, e])
    return out

print("gộp:", gop([[8, 10], [1, 3], [15, 18], [2, 6]]))

gộp: [[1, 6], [8, 10], [15, 18]]


### Kỹ thuật quét điểm (sweep line)

Biến mỗi khoảng thành hai **sự kiện**: `+1` tại điểm đầu, `−1` sau điểm cuối; sắp xếp sự kiện rồi quét, cộng dồn → biết tại mỗi thời điểm có bao nhiêu khoảng đang "mở" (ví dụ số phòng họp cần dùng).

In [3]:
def so_phong_can(intervals):
    su_kien = []
    for s, e in intervals:
        su_kien.append((s, 1))
        su_kien.append((e, -1))      # khoảng nửa mở [s, e): kết thúc trước khi cái khác bắt đầu cùng lúc
    dang_mo = best = 0
    for _, d in sorted(su_kien):
        dang_mo += d
        best = max(best, dang_mo)
    return best

print("số phòng tối thiểu cho [[0,30],[5,10],[15,20]]:", so_phong_can([[0, 30], [5, 10], [15, 20]]))

số phòng tối thiểu cho [[0,30],[5,10],[15,20]]: 2


Lỗi hay gặp: dùng `<` thay `<=` (hoặc ngược lại) — đề quy định khoảng **chạm nhau** có tính là chồng không; quên sắp xếp; gộp bằng `e` mới thay vì `max(e_cũ, e_mới)` (khoảng sau có thể nằm gọn trong khoảng trước).

## 2.2 Áp dụng vào bài này

Kết quả là một danh sách **khoảng** `[start, end]`. Các số đã cho chia đoạn `[lower, upper]` thành các khe; ta duyệt các số **đã sắp xếp** như duyệt các khoảng theo điểm đầu, và mỗi khe giữa hai số kề nhau `prev` và `cur` (với `cur − prev >= 2`) cho một khoảng thiếu `[prev + 1, cur − 1]`. Thêm hai **lính canh** `lower − 1` và `upper + 1` vào hai đầu để không phải xử lý riêng khe đầu và khe cuối.

# 3. Nền tảng

In [4]:
from typing import List
import random, time

# Với hai số nguyên a < b đứng liền nhau trong dãy đã sắp xếp,
# các số nằm HẲN giữa chúng là a+1, a+2, ..., b-1
a, b = 3, 50
print("số nằm giữa 3 và 50:", b - a - 1, "số, từ", a + 1, "đến", b - 1)
a, b = 0, 1
print("giữa 0 và 1:", b - a - 1, "số → không có khoảng trống")

số nằm giữa 3 và 50: 46 số, từ 4 đến 49
giữa 0 và 1: 0 số → không có khoảng trống


Quy tắc cốt lõi: giữa `a` và `b` (liền nhau, `a < b`) có khoảng trống **khi và chỉ khi** `b − a >= 2`, và khoảng đó là `[a + 1, b − 1]`.

# 4. Baseline: duyệt mọi số trong `[lower, upper]`

Đưa `nums` vào `set`, đi qua từng `x` từ `lower` tới `upper`, gom các số không có thành khoảng. Đúng, nhưng `upper − lower` có thể tới `2·10^9` → quá chậm.

In [5]:
def missing_brute(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    have = set(nums)
    out = []
    for x in range(lower, upper + 1):
        if x not in have:
            if out and out[-1][1] == x - 1:
                out[-1][1] = x                  # nối dài khoảng cuối
            else:
                out.append([x, x])              # mở khoảng mới
    return out

print(missing_brute([0, 1, 3, 50, 75], 0, 99))

t0 = time.time(); missing_brute([], 0, 10**6); t1 = time.time()
print(f"khoảng dài 10^6 mất {t1 - t0:.3f}s → khoảng dài 2·10^9 sẽ mất khoảng {(t1 - t0) * 2000 / 60:.0f} phút")

[[2, 2], [4, 49], [51, 74], [76, 99]]


khoảng dài 10^6 mất 0.343s → khoảng dài 2·10^9 sẽ mất khoảng 11 phút


# 5. Ý tưởng chính: chỉ nhìn vào các cặp liền kề, kèm lính canh

Các số bị thiếu chỉ có thể nằm:

1. trước `nums[0]` (từ `lower`),
2. giữa hai phần tử liền kề `nums[i]` và `nums[i+1]`,
3. sau `nums[-1]` (tới `upper`).

Ba trường hợp này **gộp làm một** nếu ta tưởng tượng thêm hai **lính canh**: `lower − 1` ở đầu và `upper + 1` ở cuối. Khi đó khoảng thiếu ở hai đầu cũng chỉ là "khoảng giữa hai số liền kề".

```text
lower − 1 = -1   [0, 1, 3, 50, 75]   upper + 1 = 100

cặp (-1, 0):   0 − (−1) = 1  → không thiếu
cặp (1, 3):    3 − 1 = 2     → thiếu [2, 2]
cặp (3, 50):   47 >= 2       → thiếu [4, 49]
cặp (50, 75):                → thiếu [51, 74]
cặp (75, 100):               → thiếu [76, 99]
```

```text
prev = lower - 1
for x in nums + [upper + 1]:
    if x - prev >= 2:
        out.append([prev + 1, x - 1])
    prev = x
```

Chỉ `O(n)`, không phụ thuộc độ dài `[lower, upper]`.

In [6]:
def missing_ranges(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    out = []
    prev = lower - 1                        # lính canh đầu
    for x in nums + [upper + 1]:            # lính canh cuối
        if x - prev >= 2:
            out.append([prev + 1, x - 1])
        prev = x
    return out

print(missing_ranges([0, 1, 3, 50, 75], 0, 99))
print(missing_ranges([-1], -1, -1), missing_ranges([], 1, 1))

[[2, 2], [4, 49], [51, 74], [76, 99]]
[] [[1, 1]]


# 6. Mô phỏng

In [7]:
def trace_missing(nums, lower, upper):
    out, prev = [], lower - 1
    print(f"lính canh: prev = lower - 1 = {prev}, thêm upper + 1 = {upper + 1} vào cuối")
    for x in nums + [upper + 1]:
        gap = x - prev
        if gap >= 2:
            out.append([prev + 1, x - 1])
            note = f"thêm [{prev + 1}, {x - 1}]"
        else:
            note = "liền nhau, không thiếu"
        print(f"   cặp ({prev:>4}, {x:>4}): hiệu = {gap:>3} → {note}")
        prev = x
    return out

print("→", trace_missing([0, 1, 3, 50, 75], 0, 99))

lính canh: prev = lower - 1 = -1, thêm upper + 1 = 100 vào cuối
   cặp (  -1,    0): hiệu =   1 → liền nhau, không thiếu
   cặp (   0,    1): hiệu =   1 → liền nhau, không thiếu
   cặp (   1,    3): hiệu =   2 → thêm [2, 2]
   cặp (   3,   50): hiệu =  47 → thêm [4, 49]
   cặp (  50,   75): hiệu =  25 → thêm [51, 74]
   cặp (  75,  100): hiệu =  25 → thêm [76, 99]
→ [[2, 2], [4, 49], [51, 74], [76, 99]]


# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def findMissingRanges(self, nums: List[int], lower: int, upper: int) -> List[List[int]]:
        out = []
        prev = lower - 1
        for x in nums + [upper + 1]:
            if x - prev >= 2:
                out.append([prev + 1, x - 1])
            prev = x
        return out

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.findMissingRanges([0, 1, 3, 50, 75], 0, 99), [[2, 2], [4, 49], [51, 74], [76, 99]])
check("Ví dụ 2", sol.findMissingRanges([-1], -1, -1), [])
check("Rỗng", sol.findMissingRanges([], 1, 1), [[1, 1]])
check("Rỗng, khoảng dài", sol.findMissingRanges([], -3, 4), [[-3, 4]])
check("Đủ hết", sol.findMissingRanges([1, 2, 3], 1, 3), [])
check("Thiếu hai đầu", sol.findMissingRanges([5], 1, 9), [[1, 4], [6, 9]])
check("Giới hạn lớn", sol.findMissingRanges([0], -10**9, 10**9), [[-10**9, -1], [1, 10**9]])

random.seed(163)
for _ in range(5000):
    lower = random.randint(-10, 10)
    upper = lower + random.randint(0, 15)
    k = random.randint(0, upper - lower + 1)
    nums = sorted(random.sample(range(lower, upper + 1), k))
    assert sol.findMissingRanges(nums, lower, upper) == missing_brute(nums, lower, upper), (nums, lower, upper)
print("5000 bộ ngẫu nhiên: khớp brute force")

Ví dụ 1          PASS  [[2, 2], [4, 49], [51, 74], [76, 99]]
Ví dụ 2          PASS  []
Rỗng             PASS  [[1, 1]]
Rỗng, khoảng dài PASS  [[-3, 4]]
Đủ hết           PASS  []
Thiếu hai đầu    PASS  [[1, 4], [6, 9]]
Giới hạn lớn     PASS  [[-1000000000, -1], [1, 1000000000]]
5000 bộ ngẫu nhiên: khớp brute force


# 8. Vì sao đúng? Độ phức tạp?

Gọi dãy mở rộng là `s = [lower − 1] + nums + [upper + 1]` — vẫn tăng ngặt. Mỗi số bị thiếu `x` thuộc `[lower, upper]` và không có trong `nums`, nên nằm **hẳn giữa** đúng một cặp liền kề `(s[i], s[i+1])`. Ngược lại, mọi số hẳn giữa một cặp liền kề đều bị thiếu. Vậy các khoảng `[s[i] + 1, s[i+1] − 1]` (khi khác rỗng) phủ đúng tập số bị thiếu, rời nhau, đã sắp xếp, và không thể gộp thêm (giữa hai khoảng liên tiếp luôn có ít nhất một số của `nums`) → **ngắn nhất**.

```text
Time:  O(n)        (n = len(nums), KHÔNG phụ thuộc upper − lower)
Space: O(1) ngoài kết quả  (nums + [upper+1] tạo một bản sao O(n); có thể tránh bằng xử lý riêng phần tử cuối)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên khoảng trống ở cuối (không thêm lính canh `upper + 1`)

In [10]:
def wrong_no_tail(nums, lower, upper):
    out, prev = [], lower - 1
    for x in nums:                                 # SAI: không xét đoạn sau nums[-1]
        if x - prev >= 2:
            out.append([prev + 1, x - 1])
        prev = x
    return out

print("Sai :", wrong_no_tail([0, 1, 3, 50, 75], 0, 99))
print("Đúng:", sol.findMissingRanges([0, 1, 3, 50, 75], 0, 99))
print("nums rỗng — sai:", wrong_no_tail([], 1, 1), "| đúng:", sol.findMissingRanges([], 1, 1))

Sai : [[2, 2], [4, 49], [51, 74]]
Đúng: [[2, 2], [4, 49], [51, 74], [76, 99]]
nums rỗng — sai: [] | đúng: [[1, 1]]


## Lỗi 2: Điều kiện `x − prev >= 1` (lệch 1)

In [11]:
def wrong_off_by_one(nums, lower, upper):
    out, prev = [], lower - 1
    for x in nums + [upper + 1]:
        if x - prev >= 1:                          # SAI: hai số liền nhau (hiệu 1) cũng bị coi là có khoảng trống
            out.append([prev + 1, x - 1])
        prev = x
    return out

print("Sai :", wrong_off_by_one([1, 2, 3], 1, 3), "| Đúng:", sol.findMissingRanges([1, 2, 3], 1, 3))

Sai : [[1, 0], [2, 1], [3, 2], [4, 3]] | Đúng: []


Hiệu bằng 1 nghĩa là hai số **liền nhau**, giữa chúng không có số nào — kết quả sai sinh ra các "khoảng" `[a, a−1]` vô nghĩa.

## Lỗi 3: Lính canh đầu là `lower` thay vì `lower − 1`

In [12]:
def wrong_sentinel(nums, lower, upper):
    out, prev = [], lower                          # SAI: coi như lower đã có mặt
    for x in nums + [upper + 1]:
        if x - prev >= 2:
            out.append([prev + 1, x - 1])
        prev = x
    return out

print("Sai :", wrong_sentinel([5], 1, 9), "| Đúng:", sol.findMissingRanges([5], 1, 9))

Sai :

 [[2, 4], [6, 9]] | Đúng: [[1, 4], [6, 9]]


Số `lower` (ở đây là `1`) bị thiếu nhưng không được đưa vào kết quả.

## Lỗi 4: Duyệt cả khoảng `[lower, upper]`

Như baseline ở mục 4: khoảng `[−10^9, 10^9]` có `2·10^9 + 1` số → mất nhiều phút (xem số đo ở mục 4), trong khi `nums` chỉ có tối đa 100 phần tử. Độ phức tạp phải phụ thuộc vào `len(nums)`, không phải độ rộng khoảng.

# 10. Mẹo nhận diện pattern

- **"Khoảng trống giữa các phần tử của dãy đã sắp xếp"** → xét từng **cặp liền kề**.
- **Trường hợp biên làm code rối** → thêm **lính canh** ở hai đầu để biên trở thành trường hợp thường.
- **Phần bù của một tập khoảng** trong `[lower, upper]` → sắp xếp, gộp, rồi lấy các khe giữa (bài tập 5).

Bài liên quan: **#228 Summary Ranges** (Easy), **#56 Merge Intervals**, **#57 Insert Interval**, **#352 Data Stream as Disjoint Intervals** (Hard), **#1272 Remove Interval**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [2, 3, 7, 10]`, `lower = 0`, `upper = 12`, liệt kê các cặp liền kề (kể cả lính canh) và kết quả.

## Bài 2 — Điền chỗ trống

In [13]:
def exercise_2_missing_ranges(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    out = []
    prev = lower - 1
    for x in nums + [upper + 1]:
        # TODO: nếu có khoảng trống giữa prev và x thì thêm vào out; cập nhật prev
        pass
    return out


# assert exercise_2_missing_ranges([0, 1, 3, 50, 75], 0, 99) == [[2, 2], [4, 49], [51, 74], [76, 99]]
# assert exercise_2_missing_ranges([-1], -1, -1) == []
# assert exercise_2_missing_ranges([], 1, 1) == [[1, 1]]

## Bài 3 — Biến thể: định dạng chuỗi (phiên bản cũ của đề)

Trả về list chuỗi: khoảng một số ghi `"a"`, khoảng nhiều số ghi `"a->b"`.

In [14]:
def exercise_3_missing_ranges_str(nums: List[int], lower: int, upper: int) -> List[str]:
    # TODO
    pass


# assert exercise_3_missing_ranges_str([0, 1, 3, 50, 75], 0, 99) == ["2", "4->49", "51->74", "76->99"]
# assert exercise_3_missing_ranges_str([], -2, -2) == ["-2"]

## Bài 4 — Biến thể: Summary Ranges (LeetCode #228)

Ngược lại với bài gốc: gom các số **có mặt** (mảng tăng ngặt) thành các khoảng liên tiếp, định dạng `"a"` hoặc `"a->b"`.

In [15]:
def exercise_4_summary_ranges(nums: List[int]) -> List[str]:
    # TODO
    pass


# assert exercise_4_summary_ranges([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
# assert exercise_4_summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
# assert exercise_4_summary_ranges([]) == []

## Bài 5 — Mini challenge: phần bù của một tập khoảng

Cho danh sách khoảng đóng `intervals` (chưa sắp xếp, có thể chồng nhau, có thể tràn ra ngoài `[lower, upper]`). Trả về các khoảng **số nguyên** trong `[lower, upper]` không bị khoảng nào phủ.

In [16]:
def exercise_5_uncovered(intervals: List[List[int]], lower: int, upper: int) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_uncovered([[5, 8], [1, 2], [7, 10]], 0, 12) == [[0, 0], [3, 4], [11, 12]]
# assert exercise_5_uncovered([[-5, 50]], 0, 12) == []
# assert exercise_5_uncovered([], 3, 4) == [[3, 4]]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_missing` sau khi tự làm.

**Bài 3**
- Mức 1: dùng lại kết quả bài gốc rồi định dạng: `str(a) if a == b else f"{a}->{b}"`.

**Bài 4**
- Mức 1: hai con trỏ: `i` ở đầu nhóm, `j` chạy chừng nào `nums[j+1] == nums[j] + 1`.

**Bài 5**
- Mức 1: sắp xếp theo điểm đầu, duy trì `covered_to` = số lớn nhất đã được phủ (bắt đầu `lower − 1`).
- Mức 2: với khoảng `[s, e]`: nếu `s > covered_to + 1` thì khe `[covered_to + 1, min(s − 1, upper)]`; rồi `covered_to = max(covered_to, e)`.
- Mức 3: cuối cùng, nếu `covered_to < upper` thì thêm `[covered_to + 1, upper]`. Nhớ chỉ giữ khe giao với `[lower, upper]`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
print("→", trace_missing([2, 3, 7, 10], 0, 12))

lính canh: prev = lower - 1 = -1, thêm upper + 1 = 13 vào cuối
   cặp (  -1,    2): hiệu =   3 → thêm [0, 1]
   cặp (   2,    3): hiệu =   1 → liền nhau, không thiếu
   cặp (   3,    7): hiệu =   4 → thêm [4, 6]
   cặp (   7,   10): hiệu =   3 → thêm [8, 9]
   cặp (  10,   13): hiệu =   3 → thêm [11, 12]
→ [[0, 1], [4, 6], [8, 9], [11, 12]]


## Đáp án Bài 2

In [18]:
def answer_2_missing_ranges(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    return Solution().findMissingRanges(nums, lower, upper)

assert answer_2_missing_ranges([0, 1, 3, 50, 75], 0, 99) == [[2, 2], [4, 49], [51, 74], [76, 99]]
assert answer_2_missing_ranges([-1], -1, -1) == []
assert answer_2_missing_ranges([], 1, 1) == [[1, 1]]
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [19]:
def answer_3_missing_ranges_str(nums: List[int], lower: int, upper: int) -> List[str]:
    return [str(a) if a == b else f"{a}->{b}" for a, b in missing_ranges(nums, lower, upper)]

assert answer_3_missing_ranges_str([0, 1, 3, 50, 75], 0, 99) == ["2", "4->49", "51->74", "76->99"]
assert answer_3_missing_ranges_str([], -2, -2) == ["-2"]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [20]:
def answer_4_summary_ranges(nums: List[int]) -> List[str]:
    out, i = [], 0
    while i < len(nums):
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
            j += 1
        out.append(str(nums[i]) if i == j else f"{nums[i]}->{nums[j]}")
        i = j + 1
    return out

assert answer_4_summary_ranges([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
assert answer_4_summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
assert answer_4_summary_ranges([]) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [21]:
def answer_5_uncovered(intervals: List[List[int]], lower: int, upper: int) -> List[List[int]]:
    out = []
    covered_to = lower - 1
    for s, e in sorted(intervals):
        if s > covered_to + 1:
            a, b = covered_to + 1, min(s - 1, upper)
            if a <= b:
                out.append([a, b])
        covered_to = max(covered_to, e)
        if covered_to >= upper:
            break
    if covered_to < upper:
        out.append([max(covered_to + 1, lower), upper])
    return out

def brute_uncovered(intervals, lower, upper):
    have = {x for s, e in intervals for x in range(s, e + 1)}
    return missing_brute(sorted(x for x in have if lower <= x <= upper), lower, upper)

assert answer_5_uncovered([[5, 8], [1, 2], [7, 10]], 0, 12) == [[0, 0], [3, 4], [11, 12]]
assert answer_5_uncovered([[-5, 50]], 0, 12) == []
assert answer_5_uncovered([], 3, 4) == [[3, 4]]
for _ in range(3000):
    lower = random.randint(-5, 5); upper = lower + random.randint(0, 12)
    ivs = []
    for _ in range(random.randint(0, 5)):
        s = random.randint(lower - 4, upper + 4); ivs.append([s, s + random.randint(0, 5)])
    assert answer_5_uncovered(ivs, lower, upper) == brute_uncovered(ivs, lower, upper), (ivs, lower, upper)
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Khoảng thiếu chỉ nằm giữa hai số liền kề — kể cả hai "lính canh" `lower − 1` và `upper + 1`. Duyệt `nums + [upper + 1]` với `prev = lower − 1`: nếu `x − prev >= 2` thì thêm `[prev + 1, x − 1]`. `O(n)`, không phụ thuộc độ rộng khoảng.

---

## Checklist trước khi nộp

- [ ] Lính canh đầu là `lower − 1`, lính canh cuối là `upper + 1`?
- [ ] Điều kiện khoảng trống là hiệu `>= 2`?
- [ ] `nums` rỗng → trả `[[lower, upper]]`?
- [ ] Không lặp qua toàn bộ `[lower, upper]`?